# Task 6 MLOps and Deployment Simulation

I would use the saved Random Forest regression model to simulate a traffic prediction deployment through a local API.

I would record the deployed model version and its evaluation metrics, validate incoming inputs, and return predicted hourly traffic volume.

I would also compare incoming feature distributions with training data to demonstrate monitoring. 

### Configure logging and check deployment packages

I would configure the execution log and check whether FastAPI, Uvicorn and HTTPX are available for the local API and its verification.

In [3]:
# Configure Logging
import logging
from pathlib import Path
from importlib.util import find_spec

import pandas as pd

PART3_DIR = Path.cwd().parent
REPO_DIR = PART3_DIR.parent

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(
            PART3_DIR / "part3_smart_city_traffic_mlops_deployment.log",
            encoding="utf-8"
        )
    ],
    force=True
)

logger = logging.getLogger(__name__)

logger.info("MLOps and deployment simulation notebook has started.")


deployment_packages = pd.DataFrame([
    {
        "package": package,
        "available": find_spec(package) is not None
    }
    for package in ["fastapi", "uvicorn", "httpx"]
])

display(deployment_packages)


logger.info("Deployment package availability check has completed.")


2026-10-11 17:34:27,661 | INFO | __main__ | MLOps and deployment simulation notebook has started.


,package,available
0,fastapi,True
1,uvicorn,True
2,httpx,True


2026-10-11 17:34:27,714 | INFO | __main__ | Deployment package availability check has completed.


### Verify deployment package imports

I would import FastAPI, Uvicorn and HTTPX and record their installed versions before creating the local prediction API.

In [4]:
import fastapi
import uvicorn
import httpx

deployment_versions = pd.DataFrame([
    {"package": "fastapi", "version": fastapi.__version__},
    {"package": "uvicorn", "version": uvicorn.__version__},
    {"package": "httpx", "version": httpx.__version__}
])

display(deployment_versions)

logger.info(
    "FastAPI %s, Uvicorn %s and HTTPX %s are imported successfully.",
    fastapi.__version__,
    uvicorn.__version__,
    httpx.__version__
)



,package,version
0,fastapi,0.142.2
1,uvicorn,0.54.0
2,httpx,0.28.1


2026-10-11 17:35:36,815 | INFO | __main__ | FastAPI 0.142.2, Uvicorn 0.54.0 and HTTPX 0.28.1 are imported successfully.


### Record the deployment model and evaluation metrics

I would identify the saved Random Forest regression v1 model and retain its validation and test metrics in a deployment manifest.

In [5]:
import json
import hashlib

deployment_dir = PART3_DIR / "deployment"
deployment_dir.mkdir(parents=True, exist_ok=True)

model_path = PART3_DIR / "models" / "random_forest_regression_v1.joblib"
config_path = PART3_DIR / "models" / "preparation_config_v1.joblib"

for artifact in [model_path, config_path]:
    if not artifact.is_file():
        raise FileNotFoundError(
            f"Required deployment artifact is missing: {artifact.name}."
        )

validation_results = pd.read_csv(
    PART3_DIR / "results" / "task3_validation_comparison.csv"
)
test_results = pd.read_csv(
    PART3_DIR / "results" / "task3_test_comparison.csv"
)

validation_row = validation_results.loc[
    validation_results["model"].eq("Random Forest")
].iloc[0]

test_row = test_results.loc[
    test_results["model"].eq("Random Forest")
].iloc[0]


deployment_manifest = {
    "model_name": "Random Forest regression",
    "model_version": "v1",
    "target": "traffic_volume",
    "prediction_units": "vehicles per hour",
    "model_file": "models/random_forest_regression_v1.joblib",
    "preparation_file": "models/preparation_config_v1.joblib",
    "model_sha256": hashlib.sha256(model_path.read_bytes()).hexdigest(),
    "preparation_sha256": hashlib.sha256(
        config_path.read_bytes()
    ).hexdigest(),
    "validation_MAE": float(validation_row["validation_MAE"]),
    "validation_R_squared": float(
        validation_row["validation_R_squared"]
    ),
    "test_MAE": float(test_row["test_MAE"]),
    "test_R_squared": float(test_row["test_R_squared"]),
    "deployment_type": "local simulation"
}

(deployment_dir / "deployment_manifest.json").write_text(
    json.dumps(deployment_manifest, indent=2, allow_nan=False) + "\n",
    encoding="utf-8"
)


display(pd.DataFrame([{
    key: deployment_manifest[key]
    for key in [
        "model_name",
        "model_version",
        "validation_MAE",
        "validation_R_squared",
        "test_MAE",
        "test_R_squared"
    ]
}]).round(4))


logger.info(
    "Saved the deployment manifest with model version, hashes and metrics."
)

,model_name,model_version,validation_MAE,validation_R_squared,test_MAE,test_R_squared
0,Random Forest regression,v1,317.5692,0.9256,250.8358,0.9448


2026-10-11 17:38:38,275 | INFO | __main__ | Saved the deployment manifest with model version, hashes and metrics.


### Load and verify the deployment model

I would verify the model and preparation file hashes against the deployment manifest, then load the reusable input preparation module from Task 5.

This ensures that the simulation uses the recorded model artifacts and the same feature preparation.

In [6]:
import importlib.util
import numpy as np


saved_manifest = json.loads(
    (deployment_dir / "deployment_manifest.json").read_text(
        encoding="utf-8"
    )
)

for artifact_path, hash_key in [
    (model_path, "model_sha256"),
    (config_path, "preparation_sha256")
]:
    actual_hash = hashlib.sha256(
        artifact_path.read_bytes()
    ).hexdigest()

    if actual_hash != saved_manifest[hash_key]:
        raise ValueError(
            f"Deployment artifact hash mismatch: {artifact_path.name}."
        )


module_path = (
    PART3_DIR / "recommendation_system" / "recommender.py"
)

spec = importlib.util.spec_from_file_location(
    "deployment_recommender",
    module_path
)
deployment_recommender = importlib.util.module_from_spec(spec)
spec.loader.exec_module(deployment_recommender)

deployment_model = deployment_recommender.recommendation_model

example_inputs = pd.read_csv(
    PART3_DIR
    / "recommendation_system"
    / "example_candidate_hours.csv"
)

prepared_inputs = (
    deployment_recommender.prepare_recommendation_inputs(
        example_inputs
    )
)

example_predictions = deployment_model.predict(prepared_inputs)
assert np.isfinite(example_predictions).all()

display(pd.DataFrame({
    "candidate_hour": pd.to_datetime(example_inputs["date_time"]),
    "predicted_traffic_volume": example_predictions
}).round(2))

logger.info(
    "Deployment artifact hashes passed and example predictions  are completed."
)


,candidate_hour,predicted_traffic_volume
0,2018-05-07 06:00:00,5692.33
1,2018-05-07 07:00:00,5475.10
2,2018-05-07 08:00:00,5871.44
3,2018-05-07 09:00:00,4956.37
4,2018-05-07 10:00:00,4204.31


2026-10-11 17:40:05,634 | INFO | __main__ | Deployment artifact hashes passed and example predictions  are completed.


### Create the simulated prediction API

I would create a health endpoint and a prediction endpoint. The prediction endpoint would validate supplied traffic conditions, apply the saved feature preparation and return hourly traffic predictions with the model version.

Invalid inputs would receive a validation error. Predictions would retain the supplied input order.

In [7]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field


class TrafficConditions(BaseModel):
    date_time: str
    temp: float = Field(ge=200, le=330, allow_inf_nan=False)
    rain_1h: float = Field(ge=0, le=9000, allow_inf_nan=False)
    snow_1h: float = Field(ge=0, allow_inf_nan=False)
    clouds_all: float = Field(ge=0, le=100, allow_inf_nan=False)
    weather_main: str
    is_holiday: int = Field(ge=0, le=1, strict=True)

class PredictionRequest(BaseModel):
    records: list[TrafficConditions] = Field(
        min_length=1,
        max_length=1000
    )
    

app = FastAPI(
    title="Smart City Traffic Prediction",
    version="1.0"
)

@app.get("/health")
def health():
    return {
        "status": "ready",
        "model_name": saved_manifest["model_name"],
        "model_version": saved_manifest["model_version"],
        "deployment_type": "local simulation"
    }


@app.post("/predict")
def predict(request: PredictionRequest):
    inputs = pd.DataFrame([
        record.model_dump()
        for record in request.records
    ])

    try:
        features = (
            deployment_recommender.prepare_recommendation_inputs(
                inputs
            )
        )
    except (ValueError, TypeError, KeyError):
        logger.warning("Prediction request failed input validation.")
        raise HTTPException(
            status_code=422,
            detail=(
                "Check timestamps, weather categories and required "
                "traffic conditions."
            )
        ) from None

    try:
        predictions = deployment_model.predict(features)

        if not np.isfinite(predictions).all():
            raise ValueError("Non-finite model predictions.")

        records = [
            {
                "date_time": str(timestamp),
                "predicted_traffic_volume": float(prediction)
            }
            for timestamp, prediction in zip(
                pd.to_datetime(inputs["date_time"]),
                predictions
            )
        ]

        logger.info(
            "Completed prediction request for %s records.",
            len(records)
        )

        
        return {
            "model_version": saved_manifest["model_version"],
            "prediction_units": saved_manifest["prediction_units"],
            "predictions": records
        }

    except Exception:
        logger.error("Prediction request failed during model inference.")
        raise HTTPException(
            status_code=500,
            detail="Model inference failed."
        ) from None

logger.info("Created the health and prediction API endpoints.")


2026-10-11 17:41:23,759 | INFO | __main__ | Created the health and prediction API endpoints.


### Verify API predictions and input validation

I would test the health and prediction endpoints without starting a network server. 

Invalid weather, duplicate timestamps, missing fields and invalid cloud cover should receive HTTP 422 responses.

In [8]:
from copy import deepcopy
from fastapi.testclient import TestClient

api_payload = {"records": example_inputs.to_dict(orient="records")}

api_check_records = []

with TestClient(app) as client:
    health_response = client.get("/health")
    assert health_response.status_code == 200
    assert health_response.json()["model_version"] == "v1"

    api_check_records.append({
        "check": "Health endpoint",
        "HTTP_status": health_response.status_code,
        "result": "PASS"
    })

    
    prediction_response = client.post("/predict", json=api_payload)
    assert prediction_response.status_code == 200

    api_prediction_result = prediction_response.json()
    assert api_prediction_result["model_version"] == "v1"
    assert api_prediction_result["prediction_units"] == "vehicles per hour"

    api_predictions = pd.DataFrame(
        api_prediction_result["predictions"]
    )

    np.testing.assert_allclose(
        api_predictions["predicted_traffic_volume"],
        example_predictions
    )
    assert (
        pd.to_datetime(api_predictions["date_time"]).tolist()
        == pd.to_datetime(example_inputs["date_time"]).tolist()
    )

    api_check_records.append({
        "check": "Prediction values and input order",
        "HTTP_status": prediction_response.status_code,
        "result": "PASS"
    })

    invalid_weather = deepcopy(api_payload)
    invalid_weather["records"][0]["weather_main"] = "UnknownWeather"

    duplicate_time = deepcopy(api_payload)
    duplicate_time["records"][1]["date_time"] = (
        duplicate_time["records"][0]["date_time"]
    )

    missing_field = deepcopy(api_payload)
    del missing_field["records"][0]["temp"]

    invalid_clouds = deepcopy(api_payload)
    invalid_clouds["records"][0]["clouds_all"] = 150

    for check_name, invalid_payload in [
        ("Unknown weather category", invalid_weather),
        ("Duplicate timestamp", duplicate_time),
        ("Missing temperature", missing_field),
        ("Cloud cover above 100", invalid_clouds)
    ]:
        response = client.post("/predict", json=invalid_payload)
        assert response.status_code == 422

        api_check_records.append({
            "check": check_name,
            "HTTP_status": response.status_code,
            "result": "PASS"
        })

api_validation_checks = pd.DataFrame(api_check_records)

display(api_predictions.round(2))
display(api_validation_checks.style.hide(axis="index"))

logger.info(
    "API health, prediction equivalence and input validation checks passed."
)


C:\Users\mitha\anaconda3\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa
2026-10-11 17:43:09,140 | INFO | httpx | HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"
2026-10-11 17:43:09,286 | INFO | __main__ | Completed prediction request for 5 records.
2026-10-11 17:43:09,296 | INFO | httpx | HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"
2026-10-11 17:43:09,314 | WARNING | __main__ | Prediction request failed input validation.
2026-10-11 17:43:09,316 | INFO | httpx | HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"
2026-10-11 17:43:09,325 | WARNING | __main__ | Prediction request failed input validation.
2026-10-11 17:43:09,328 | INFO | httpx | HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"
2026-10-11 17:43:09,333 | INFO |

,date_time,predicted_traffic_volume
0,2018-05-07 06:00:00,5692.33
1,2018-05-07 07:00:00,5475.10
2,2018-05-07 08:00:00,5871.44
3,2018-05-07 09:00:00,4956.37
4,2018-05-07 10:00:00,4204.31


check,HTTP_status,result
Health endpoint,200,PASS
Prediction values and input order,200,PASS
Unknown weather category,422,PASS
Duplicate timestamp,422,PASS
Missing temperature,422,PASS
Cloud cover above 100,422,PASS


2026-10-11 17:43:10,612 | INFO | __main__ | API health, prediction equivalence and input validation checks passed.


### Save API verification results

I would save the example request, API response and validation checks as deployment evidence. 

In [9]:
results_dir = PART3_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)


for filename, content in [
    ("example_prediction_request.json", api_payload),
    ("example_prediction_response.json", api_prediction_result)
]:
    (deployment_dir / filename).write_text(
        json.dumps(content, indent=2, allow_nan=False) + "\n",
        encoding="utf-8"
    )

api_predictions.to_csv(
    results_dir / "deployment_example_predictions.csv",
    index=False
)

api_validation_checks.to_csv(
    results_dir / "deployment_api_validation_checks.csv",
    index=False
)

logger.info(
    "Saved the API example request, response, predictions and validation checks."
)


2026-10-11 17:46:13,957 | INFO | __main__ | Saved the API example request, response, predictions and validation checks.


### Prepare the training reference for feature monitoring

I would read the numeric feature means and standard deviations from the saved preprocessing scaler, which was fitted on training data.

Monitoring would compare batch means for temperature, rain, cloud cover and hour against this reference. A 

In [10]:
numeric_scaler = (
    deployment_model.named_steps["preprocessor"]
    .named_transformers_["numeric"]
)


numeric_feature_names = (
    deployment_recommender.recommendation_config["numeric_features"]
)

assert len(numeric_feature_names) == len(numeric_scaler.mean_)

training_reference = pd.DataFrame({
    "feature": numeric_feature_names,
    "training_mean": numeric_scaler.mean_,
    "training_std": np.sqrt(numeric_scaler.var_)
}).set_index("feature")

monitored_features = ["temp", "rain_1h", "clouds_all", "hour"]


monitoring_reference = training_reference.loc[
    monitored_features
].copy()

assert np.isfinite(monitoring_reference.to_numpy()).all()
assert monitoring_reference["training_std"].gt(0).all()

monitoring_config = {
    "model_version": saved_manifest["model_version"],
    "method": "absolute mean shift divided by training standard deviation",
    "alert_threshold": 1.0,
    "minimum_batch_size": 100,
    "reference_source": "saved training preprocessing scaler",
    "features": monitoring_reference.reset_index().to_dict(
        orient="records"
    )
}


(deployment_dir / "monitoring_reference.json").write_text(
    json.dumps(monitoring_config, indent=2, allow_nan=False) + "\n",
    encoding="utf-8"
)

display(
    monitoring_reference.reset_index()
    .style.hide(axis="index")
    .format({
        "training_mean": "{:.4f}",
        "training_std": "{:.4f}"
    })
)


logger.info(
    "Saved training feature statistics and demonstration monitoring settings."
)


feature,training_mean,training_std
temp,280.9026,12.5895
rain_1h,0.1481,1.1243
clouds_all,49.8944,38.8485
hour,11.3978,6.9523


2026-10-11 17:48:07,932 | INFO | __main__ | Saved training feature statistics and demonstration monitoring settings.


In [11]:
def monitor_feature_means(candidates):
    features = (
        deployment_recommender.prepare_recommendation_inputs(
            candidates
        )
    )

    minimum_size = monitoring_config["minimum_batch_size"]
    if len(features) < minimum_size:
        raise ValueError(
            f"Monitoring requires at least {minimum_size} records."
        )

    threshold = monitoring_config["alert_threshold"]
    records = []

    for reference in monitoring_config["features"]:
        feature = reference["feature"]
        batch_mean = float(features[feature].mean())

        mean_shift = abs(
            batch_mean - reference["training_mean"]
        ) / reference["training_std"]

        records.append({
            "feature": feature,
            "training_mean": reference["training_mean"],
            "batch_mean": batch_mean,
            "standardised_mean_shift": mean_shift,
            "threshold": threshold,
            "status": "ALERT" if mean_shift > threshold else "PASS"
        })

    checks = pd.DataFrame(records)
    overall_status = (
        "ALERT"
        if checks["status"].eq("ALERT").any()
        else "PASS"
    )

    if overall_status == "ALERT":
        logger.warning(
            "Feature mean monitoring returned ALERT for %s records.",
            len(features)
        )
    else:
        logger.info(
            "Feature mean monitoring returned PASS for %s records.",
            len(features)
        )

    return checks, overall_status


logger.info("Created the feature mean monitoring function.")

2026-10-11 17:48:56,817 | INFO | __main__ | Created the feature mean monitoring function.


### Demonstrate PASS and ALERT monitoring outcomes

I would create a synthetic batch with feature means close to the training reference, then increase its temperature by two training standard deviations.

The first batch should return PASS, and the shifted batch should return ALERT. 

In [12]:
reference_like_batch = pd.DataFrame({
    "date_time": pd.date_range(
        "2018-05-01 00:00:00",
        periods=240,
        freq="h"
    ),
    "temp": monitoring_reference.loc["temp", "training_mean"],
    "rain_1h": monitoring_reference.loc["rain_1h", "training_mean"],
    "snow_1h": 0.0,
    "clouds_all": monitoring_reference.loc[
        "clouds_all", "training_mean"
    ],
    "weather_main": "Clouds",
    "is_holiday": 0
})


shifted_batch = reference_like_batch.copy()
shifted_batch["temp"] += (
    2 * monitoring_reference.loc["temp", "training_std"]
)

normal_checks, normal_status = monitor_feature_means(
    reference_like_batch
)
shifted_checks, shifted_status = monitor_feature_means(
    shifted_batch
)

assert normal_status == "PASS"
assert shifted_status == "ALERT"
assert (
    shifted_checks.set_index("feature").loc["temp", "status"]
    == "ALERT"
)


monitoring_results = pd.concat([
    normal_checks.assign(batch="Reference-like synthetic batch"),
    shifted_checks.assign(batch="Temperature-shifted synthetic batch")
], ignore_index=True)

display(
    monitoring_results[
        [
            "batch",
            "feature",
            "training_mean",
            "batch_mean",
            "standardised_mean_shift",
            "status"
        ]
    ].style.hide(axis="index").format({
        "training_mean": "{:.4f}",
        "batch_mean": "{:.4f}",
        "standardised_mean_shift": "{:.4f}"
    })
)

monitoring_results.to_csv(
    results_dir / "deployment_monitoring_results.csv",
    index=False
)

logger.info(
    "Synthetic monitoring checks produced the expected PASS and ALERT outcomes."
)


2026-10-11 17:50:26,838 | INFO | __main__ | Feature mean monitoring returned PASS for 240 records.
2026-10-11 17:50:26,856 | WARNING | __main__ | Feature mean monitoring returned ALERT for 240 records.


batch,feature,training_mean,batch_mean,standardised_mean_shift,status
Reference-like synthetic batch,temp,280.9026,280.9026,0.0000,PASS
Reference-like synthetic batch,rain_1h,0.1481,0.1481,0.0000,PASS
Reference-like synthetic batch,clouds_all,49.8944,49.8944,0.0000,PASS
Reference-like synthetic batch,hour,11.3978,11.5000,0.0147,PASS
Temperature-shifted synthetic batch,temp,280.9026,306.0817,2.0000,ALERT
Temperature-shifted synthetic batch,rain_1h,0.1481,0.1481,0.0000,PASS
Temperature-shifted synthetic batch,clouds_all,49.8944,49.8944,0.0000,PASS
Temperature-shifted synthetic batch,hour,11.3978,11.5000,0.0147,PASS


2026-10-11 17:50:26,928 | INFO | __main__ | Synthetic monitoring checks produced the expected PASS and ALERT outcomes.


### Verify the minimum monitoring batch size

I would confirm that a batch with fewer than 100 records is rejected rather than assigned PASS or ALERT.

In [13]:
try:
    monitor_feature_means(example_inputs)
except ValueError as error:
    assert str(error) == "Monitoring requires at least 100 records."

    monitoring_size_check = pd.DataFrame([{
        "check": "Monitoring batch below 100 records",
        "result": "PASS",
        "message": str(error)
    }])
else:
    raise AssertionError(
        "An insufficient monitoring batch was accepted."
    )


display(
    monitoring_size_check.style.hide(axis="index")
)

monitoring_size_check.to_csv(
    results_dir / "deployment_monitoring_size_check.csv",
    index=False
)

logger.info("Minimum monitoring batch size rejection check passed.")


check,result,message
Monitoring batch below 100 records,PASS,Monitoring requires at least 100 records.


2026-10-11 17:51:40,108 | INFO | __main__ | Minimum monitoring batch size rejection check passed.


### Save the reusable deployment application

I would save the verified API and monitoring functions in deployment/app.py. 

In [14]:
import ast
import textwrap

definition_names = {
    "TrafficConditions",
    "PredictionRequest",
    "health",
    "predict",
    "monitor_feature_means"
}



definition_sources = {}

for cell_source in In:
    try:
        tree = ast.parse(cell_source)
    except SyntaxError:
        continue

    for node in tree.body:
        if isinstance(node, (ast.ClassDef, ast.FunctionDef)):
            if node.name in definition_names:
                first_line = min(
                    [node.lineno]
                    + [
                        decorator.lineno
                        for decorator in node.decorator_list
                    ]
                )
                definition_sources[node.name] = "\n".join(
                    cell_source.splitlines()[
                        first_line - 1:node.end_lineno
                    ]
                )


assert definition_names == set(definition_sources)

deployment_header = textwrap.dedent('''
    """Local traffic prediction and feature monitoring simulation."""

    import hashlib
    import importlib.util
    import json
    import logging
    from pathlib import Path

    import numpy as np
    import pandas as pd
    from fastapi import FastAPI, HTTPException
    from pydantic import BaseModel, Field

    logger = logging.getLogger(__name__)
    PART3_DIR = Path(__file__).resolve().parent.parent
    deployment_dir = PART3_DIR / "deployment"

    saved_manifest = json.loads(
        (deployment_dir / "deployment_manifest.json").read_text(
            encoding="utf-8"
        )
    )
    monitoring_config = json.loads(
        (deployment_dir / "monitoring_reference.json").read_text(
            encoding="utf-8"
        )
    )

    for file_key, hash_key in [
        ("model_file", "model_sha256"),
        ("preparation_file", "preparation_sha256")
    ]:
        artifact_path = PART3_DIR / saved_manifest[file_key]
        actual_hash = hashlib.sha256(
            artifact_path.read_bytes()
        ).hexdigest()
        if actual_hash != saved_manifest[hash_key]:
            raise ValueError("Deployment artifact hash mismatch.")

    spec = importlib.util.spec_from_file_location(
        "deployment_recommender",
        PART3_DIR / "recommendation_system" / "recommender.py"
    )
    deployment_recommender = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(deployment_recommender)
    deployment_model = deployment_recommender.recommendation_model

    app = FastAPI(
        title="Smart City Traffic Prediction",
        version="1.0"
    )
''').lstrip()

deployment_source = deployment_header + "\n\n" + "\n\n".join(
    definition_sources[name]
    for name in [
        "TrafficConditions",
        "PredictionRequest",
        "health",
        "predict",
        "monitor_feature_means"
    ]
) + "\n"

compile(deployment_source, "app.py", "exec")

(deployment_dir / "app.py").write_text(
    deployment_source,
    encoding="utf-8"
)

logger.info(
    "Saved the reusable API and monitoring functions to deployment/app.py."
)


2026-10-11 17:52:55,268 | INFO | __main__ | Saved the reusable API and monitoring functions to deployment/app.py.


### Verify the saved deployment application

I would import deployment/app.py and check its health endpoint, prediction values and monitoring outcomes against the notebook results.

In [16]:
import sys

saved_app_spec = importlib.util.spec_from_file_location(
    "saved_traffic_deployment",
    deployment_dir / "app.py"
)
saved_app_module = importlib.util.module_from_spec(saved_app_spec)

sys.modules[saved_app_spec.name] = saved_app_module
saved_app_spec.loader.exec_module(saved_app_module)

with TestClient(saved_app_module.app) as client:
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["model_version"] == "v1"

    response = client.post("/predict", json=api_payload)
    assert response.status_code == 200
    saved_response = response.json()

assert (
    saved_response["model_version"]
    == api_prediction_result["model_version"]
)
assert (
    saved_response["prediction_units"]
    == api_prediction_result["prediction_units"]
)

saved_predictions = pd.DataFrame(saved_response["predictions"])
expected_predictions = pd.DataFrame(
    api_prediction_result["predictions"]
)

assert (
    saved_predictions["date_time"].tolist()
    == expected_predictions["date_time"].tolist()
)

np.testing.assert_allclose(
    saved_predictions["predicted_traffic_volume"],
    expected_predictions["predicted_traffic_volume"],
    rtol=1e-10,
    atol=1e-8
)

saved_normal_checks, saved_normal_status = (
    saved_app_module.monitor_feature_means(reference_like_batch)
)
saved_shifted_checks, saved_shifted_status = (
    saved_app_module.monitor_feature_means(shifted_batch)
)

pd.testing.assert_frame_equal(saved_normal_checks, normal_checks)
pd.testing.assert_frame_equal(saved_shifted_checks, shifted_checks)
assert saved_normal_status == "PASS"
assert saved_shifted_status == "ALERT"

logger.info(
    "Saved deployment application passed API and monitoring equivalence checks."
)


2026-10-11 17:55:01,455 | INFO | httpx | HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"
2026-10-11 17:55:01,539 | INFO | saved_traffic_deployment | Completed prediction request for 5 records.
2026-10-11 17:55:01,545 | INFO | httpx | HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"
2026-10-11 17:55:01,567 | INFO | saved_traffic_deployment | Feature mean monitoring returned PASS for 240 records.
2026-10-11 17:55:01,581 | WARNING | saved_traffic_deployment | Feature mean monitoring returned ALERT for 240 records.
2026-10-11 17:55:01,592 | INFO | __main__ | Saved deployment application passed API and monitoring equivalence checks.


### Verify requests to the running local server

I would send HTTP requests to the running Uvicorn server. The health endpoint should identify model v1, predictions should match direct inference, and invalid cloud cover should receive HTTP 422.

In [17]:
with httpx.Client(
    base_url="http://127.0.0.1:8000",
    timeout=30.0,
    trust_env=False
) as client:
    live_health = client.get("/health")
    assert live_health.status_code == 200
    assert live_health.json()["model_version"] == "v1"

    live_prediction = client.post("/predict", json=api_payload)
    assert live_prediction.status_code == 200

    live_result = live_prediction.json()
    assert live_result["model_version"] == "v1"
    assert live_result["prediction_units"] == "vehicles per hour"

    live_predictions = pd.DataFrame(live_result["predictions"])

    np.testing.assert_allclose(
        live_predictions["predicted_traffic_volume"],
        example_predictions,
        rtol=1e-10,
        atol=1e-8
    )
    assert (
        pd.to_datetime(live_predictions["date_time"]).tolist()
        == pd.to_datetime(example_inputs["date_time"]).tolist()
    )

    live_invalid = client.post("/predict", json=invalid_clouds)
    assert live_invalid.status_code == 422

live_http_checks = pd.DataFrame([
    {
        "check": "Running server health",
        "HTTP_status": live_health.status_code,
        "result": "PASS"
    },
    {
        "check": "Running server predictions and input order",
        "HTTP_status": live_prediction.status_code,
        "result": "PASS"
    },
    {
        "check": "Running server rejects invalid cloud cover",
        "HTTP_status": live_invalid.status_code,
        "result": "PASS"
    }
])



display(live_predictions.round(2))
display(live_http_checks.style.hide(axis="index"))

live_http_checks.to_csv(
    results_dir / "deployment_live_http_checks.csv",
    index=False
)

(deployment_dir / "live_prediction_response.json").write_text(
    json.dumps(live_result, indent=2, allow_nan=False) + "\n",
    encoding="utf-8"
)



logger.info(
    "Running local server passed health, prediction and validation checks."
)

2026-10-11 20:12:33,525 | INFO | httpx | HTTP Request: GET http://127.0.0.1:8000/health "HTTP/1.1 200 OK"
2026-10-11 20:12:33,718 | INFO | httpx | HTTP Request: POST http://127.0.0.1:8000/predict "HTTP/1.1 200 OK"
2026-10-11 20:12:33,758 | INFO | httpx | HTTP Request: POST http://127.0.0.1:8000/predict "HTTP/1.1 422 Unprocessable Content"


,date_time,predicted_traffic_volume
0,2018-05-07 06:00:00,5692.33
1,2018-05-07 07:00:00,5475.10
2,2018-05-07 08:00:00,5871.44
3,2018-05-07 09:00:00,4956.37
4,2018-05-07 10:00:00,4204.31


check,HTTP_status,result
Running server health,200,PASS
Running server predictions and input order,200,PASS
Running server rejects invalid cloud cover,422,PASS


2026-10-11 20:12:33,835 | INFO | __main__ | Running local server passed health, prediction and validation checks.


### Task 6 results and limitations

Th local API served the saved Random Forest regression v1 model. Its deployment manifest recorded artifact hashes and existing validation and test metrics. Hash verification passed before loading the model.

In-process API checks and requests to the runing Uvicorn server confirmed health reesponses, prediction values, input order and rejection of invalid inputs. The reusable application also reproduced the notebook monitoring results.

Feature monitoring used training means and standard deviations for temperature, rain, cloud cover and hour. A reference-like synthetic batch returned PASS, while a temperature shift of two training standard deviations triiggered ALERT. Batches below 100 records were rejected.

This is a local deployment simulation. Monitoring thresholds require calibration for operational use. Mean shifts do not detect every distribution change, and PASS does not establish unchanged prediction accuracy. Measuring prediction error on incoming traffic would require observed traffic volumes.

### Save deployment run instructions

I would document how to start the local API, submit the example request and interpret the monitoring outcomes.

In [18]:
deployment_readme = """# Smart City Traffic Deployment Simulation

The application serves Random Forest regression v1 using the saved model
and preparation configuration. Artifact hashes are checked on import.

## Start the API

From the repository root, using the project Python environment:

```cmd
python -m uvicorn app:app --app-dir capstone_part3/deployment --host 127.0.0.1 --port 8000
```

Keep the terminal running. Stop the server with Ctrl+C.

## Endpoints

- GET /health returns readiness and model version.
- POST /predict accepts a JSON object containing a records list.
- Interactive API documentation: http://127.0.0.1:8000/docs

Use example_prediction_request.json as the example request body.
Each record requires date_time, temp, rain_1h, snow_1h, clouds_all,
weather_main and is_holiday.

Temperature is in kelvin, rain and snow are in millimetres, cloud cover
is a percentage, and is_holiday is an integer flag of 0 or 1.
Timestamps must be unique, on the hour and expressed as corridor local
time without a timezone offset. Requests accept 1 to 1000 records.

Predictions retain input order and report traffic volume in vehicles
per hour. Invalid inputs return HTTP 422.

## Monitoring

monitor_feature_means is a Python function in app.py. It is not an HTTP
endpoint. It accepts a candidate DataFrame with the same input columns.

Monitoring compares batch means for temperature, rain, cloud cover and
hour with statistics from the training preprocessing scaler. Batches
require at least 100 records.

An absolute mean shift above one training standard deviation triggers
ALERT. Otherwise, the selected checks return PASS. The threshold is for
demonstration and requires operational calibration.

Synthetic examples verified PASS and ALERT behavior. Mean checks do
not detect every distribution change or measure prediction accuracy.
Accuracy monitoring requires observed traffic volumes.

## Evidence

deployment_manifest.json records model version, hashes and evaluation metrics.
monitoring_reference.json records training statistics and monitoring settings.
Example API requests and responses are stored in this folder.
Verification tables are stored in the Part 3 results folder.

## Scope

This is a local simulation using a model trained on historical traffic
records. It does not provide live traffic information or journey times.
MLflow tracking from Task 4 retains the existing model evaluations;
this task does not retrain or register a new model.
"""

(deployment_dir / "README.md").write_text(
    deployment_readme,
    encoding="utf-8"
)

logger.info("Saved deployment run instructions to deployment/README.md.")

2026-10-11 20:16:56,336 | INFO | __main__ | Saved deployment run instructions to deployment/README.md.
